<div style="border: 5px solid black; padding: 20px; border-radius: 6px;">

## **Course:** DSC640 - Data Presentation and Visualization
## **Name:** Tim Hollis
## **Assignment:** Week 7&8 Exercise
## **Date:** July 30, 2026

---

**Reference**

New York State Gaming Commission. (2026). *Lottery Mega Millions winning numbers: Beginning 2002* [Data set]. data.ny.gov. https://data.ny.gov/Government-Finance/Lottery-Mega-Millions-Winning-Numbers-Beginning-20/5xaw-6ayf

New York State Gaming Commission. (2026). *Lottery Pick 10 winning numbers: Beginning 1987* [Data set]. data.ny.gov. https://data.ny.gov/Government-Finance/Lottery-Pick-10-Winning-Numbers-Beginning-1987/bycu-cw7c

New York State Gaming Commission. (2026). *Lottery Powerball winning numbers: Beginning 2010* [Data set]. data.ny.gov. https://data.ny.gov/Government-Finance/Lottery-Powerball-Winning-Numbers-Beginning-2010/d6yy-54nr

</div>

### **Initial Setup**

In [1]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from math import comb
from IPython.display import HTML, display

# HTML/PDF Rendering Format
display(HTML('''
<style>
    pre, code, div.highlight pre, div.output_area pre {
        white-space: pre-wrap !important;
        word-break: break-word;
    }
    div.output_subarea { page-break-inside: avoid; }
    div.jp-MarkdownOutput { page-break-inside: avoid; }
    div.cell { page-break-inside: avoid; }
</style>
'''))

# Colorblind-safe palette
ROYAL_BLUE = '#4169E1'
PURPLE = '#4B0082'
FOREST_GREEN = '#228B22'
AMBER = '#E69F00'
LIGHT_GRAY = '#D3D3D3'
DARK_GRAY = '#4D4D4D'
PALETTE = [ROYAL_BLUE, PURPLE, FOREST_GREEN, AMBER]

# Global visualization formatting
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['font.size'] = 11
sns.set_palette(PALETTE)

# Load the three lottery datasets
DATA_DIR = Path('.')
mega_df = pd.read_csv(DATA_DIR / 'MegaMillions.csv')
pick10_df = pd.read_csv(DATA_DIR / 'Pick10.csv')
power_df = pd.read_csv(DATA_DIR / 'Powerball.csv')

print('🚀 Setup complete, all systems go')
print(f'🎰 Mega Millions loaded: {len(mega_df):,} drawings')
print(f'🔟 Pick 10 loaded: {len(pick10_df):,} drawings')
print(f'⚡ Powerball loaded: {len(power_df):,} drawings')

🚀 Setup complete, all systems go
🎰 Mega Millions loaded: 2,285 drawings
🔟 Pick 10 loaded: 13,570 drawings
⚡ Powerball loaded: 1,617 drawings


### **Overview**

The story is completed, using three New York State lottery datasets: Mega Millions (2002-present), Pick 10 (1987-present), and Powerball (2010-present). The Python analysis below cleans and reshapes the raw drawing data, examines number frequency and expected value, and exports tidy datasets for the six Power BI visualizations embedded later in this notebook. The story targets a general public audience of lottery players.

### **Data Cleaning and Parsing**

1. Convert draw dates to datetime and extract the year
2. Split the space-separated winning number strings into individual integer columns
3. Separate main numbers from bonus balls (Mega Ball, Powerball)
4. Verify no missing or malformed drawings

In [2]:
# Data Cleaning and Parsing
def split_numbers(series):
    """Splits a space-separated winning numbers string into a DataFrame of ints."""
    return series.str.strip().str.split(expand=True).astype(int)


# Mega Millions: 5 main numbers + separate Mega Ball column
mega_df['Draw Date'] = pd.to_datetime(mega_df['Draw Date'])
mega_df['Year'] = mega_df['Draw Date'].dt.year
mega_main = split_numbers(mega_df['Winning Numbers'])
mega_main.columns = [f'Num{i+1}' for i in range(mega_main.shape[1])]
mega_df = pd.concat([mega_df, mega_main], axis=1)

# Powerball: 6 numbers in the string, the last one is the Powerball
power_df['Draw Date'] = pd.to_datetime(power_df['Draw Date'])
power_df['Year'] = power_df['Draw Date'].dt.year
power_nums = split_numbers(power_df['Winning Numbers'])
power_nums.columns = [f'Num{i+1}' for i in range(5)] + ['Powerball']
power_df = pd.concat([power_df, power_nums], axis=1)

# Pick 10: 20 numbers per drawing, no bonus ball
pick10_df['Draw Date'] = pd.to_datetime(pick10_df['Draw Date'])
pick10_df['Year'] = pick10_df['Draw Date'].dt.year
pick10_nums = split_numbers(pick10_df['Winning Numbers'])
pick10_nums.columns = [f'Num{i+1}' for i in range(pick10_nums.shape[1])]
pick10_df = pd.concat([pick10_df, pick10_nums], axis=1)

# Verify the parse
num_cols_check = [c for c in mega_df.columns if c.startswith('Num')]
missing_total = (
    mega_df[num_cols_check].isna().sum().sum() +
    power_df[[c for c in power_df.columns if c.startswith('Num')]].isna().sum().sum() +
    pick10_df[[c for c in pick10_df.columns if c.startswith('Num')]].isna().sum().sum()
)

print(
    f'🎰 Mega Millions parsed: {mega_main.shape[1]} main numbers per draw, years {mega_df["Year"].min()}-{mega_df["Year"].max()}')
print(
    f'⚡ Powerball parsed: 5 main + 1 Powerball per draw, years {power_df["Year"].min()}-{power_df["Year"].max()}')
print(
    f'🔟 Pick 10 parsed: {pick10_nums.shape[1]} numbers per draw, years {pick10_df["Year"].min()}-{pick10_df["Year"].max()}')
print(f'🧹 Missing values across all parsed number columns: {missing_total:,}')

🎰 Mega Millions parsed: 5 main numbers per draw, years 2002-2024
⚡ Powerball parsed: 5 main + 1 Powerball per draw, years 2010-2024
🔟 Pick 10 parsed: 20 numbers per draw, years 1987-2024
🧹 Missing values across all parsed number columns: 0


### **Number Frequency Analysis**

1. Reshape each game into long format with one row per drawn number
2. Count how often each number has been drawn across the full history of each game
3. Build per-year frequency tables to support the small multiples visual
4. Compare observed frequencies against the expected uniform distribution

In [3]:
# Number Frequency Analysis
def melt_numbers(df, num_cols, game_name):
    """Reshapes wide number columns into long format: one row per drawn number."""
    long_df = df.melt(
        id_vars=['Draw Date', 'Year'],
        value_vars=num_cols,
        var_name='Position',
        value_name='Number'
    )
    long_df['Game'] = game_name
    return long_df


mega_cols = [f'Num{i+1}' for i in range(5)]
power_cols = [f'Num{i+1}' for i in range(5)]
pick10_cols = [f'Num{i+1}' for i in range(20)]

mega_long = melt_numbers(mega_df, mega_cols, 'Mega Millions')
power_long = melt_numbers(power_df, power_cols, 'Powerball')
pick10_long = melt_numbers(pick10_df, pick10_cols, 'Pick 10')

all_long = pd.concat([mega_long, power_long, pick10_long], ignore_index=True)

# Overall frequency per number per game
freq_overall = (
    all_long.groupby(['Game', 'Number'])
    .size()
    .reset_index(name='Times Drawn')
)

# Per-year frequency for small multiples
freq_yearly = (
    all_long.groupby(['Game', 'Year', 'Number'])
    .size()
    .reset_index(name='Times Drawn')
)

# Expected draws per number if the game were perfectly uniform
expected = {}
for game, group in all_long.groupby('Game'):
    n_pool = group['Number'].max()
    expected[game] = len(group) / n_pool
    top = freq_overall[freq_overall['Game'] == game].nlargest(1, 'Times Drawn')
    print(
        f'🎯 {game}: pool of {n_pool} numbers, expected {expected[game]:,.1f} draws each, hottest number is {top["Number"].values[0]} at {top["Times Drawn"].values[0]:,} draws')

print(f'📊 Long-format rows built: {len(all_long):,}')

🎯 Mega Millions: pool of 75 numbers, expected 152.3 draws each, hottest number is 31 at 219 draws
🎯 Pick 10: pool of 80 numbers, expected 3,392.5 draws each, hottest number is 58 at 3,541 draws
🎯 Powerball: pool of 69 numbers, expected 117.2 draws each, hottest number is 36 at 145 draws
📊 Long-format rows built: 290,910


### **Matrix Change Diagnostic**

1. Identify the highest number drawn in each year for each game
2. Locate the years where the number pool expanded or contracted
3. Assess whether lifetime frequency counts are comparable across the full history

In [4]:
# Matrix Change Diagnostic

pool_by_year = (
    all_long.groupby(['Game', 'Year'])['Number']
    .max()
    .reset_index(name='Highest Number Drawn')
)

for game in ['Mega Millions', 'Powerball', 'Pick 10']:
    game_pool = pool_by_year[pool_by_year['Game'] == game]
    changes = game_pool[game_pool['Highest Number Drawn'].diff() != 0]
    summary = ', '.join(
        f'{int(row.Year)}: {int(row._3)}' for row in changes.itertuples()
    )
    print(f'🔧 {game} pool ceiling by year of change -> {summary}')

print(f'📐 Distinct pool ceilings observed: Mega Millions - {pool_by_year[pool_by_year["Game"] == "Mega Millions"]["Highest Number Drawn"].nunique()}, Powerball - {pool_by_year[pool_by_year["Game"] == "Powerball"]["Highest Number Drawn"].nunique()}, Pick 10 - {pool_by_year[pool_by_year["Game"] == "Pick 10"]["Highest Number Drawn"].nunique()}')

🔧 Mega Millions pool ceiling by year of change -> 2002: 52, 2005: 56, 2013: 75, 2018: 70
🔧 Powerball pool ceiling by year of change -> 2010: 59, 2015: 69
🔧 Pick 10 pool ceiling by year of change -> 1987: 80
📐 Distinct pool ceilings observed: Mega Millions - 4, Powerball - 2, Pick 10 - 1


### **Current Matrix Filtering**

1. Restrict Mega Millions to drawings from October 31, 2017 forward (5/70 matrix)
2. Restrict Powerball to drawings from October 4, 2015 forward (5/69 matrix)
3. Retain the full Pick 10 history, which has used a 1-80 pool since 1987
4. Recalculate frequency and expected values on the comparable subsets

In [5]:
# Current Matrix Filtering
MEGA_MATRIX_START = pd.Timestamp('2017-10-31')
POWER_MATRIX_START = pd.Timestamp('2015-10-04')

matrix_filters = {
    'Mega Millions': MEGA_MATRIX_START,
    'Powerball': POWER_MATRIX_START,
    'Pick 10': pd.Timestamp('1987-01-01')
}

all_long['Matrix Start'] = all_long['Game'].map(matrix_filters)
current_long = all_long[all_long['Draw Date']
                        >= all_long['Matrix Start']].copy()
current_long = current_long.drop(columns=['Matrix Start'])

freq_current = (
    current_long.groupby(['Game', 'Number'])
    .size()
    .reset_index(name='Times Drawn')
)

# Expected value and chance band under a uniform model
band_rows = []
for game, group in current_long.groupby('Game'):
    n_pool = group['Number'].max()
    n_draws = group['Draw Date'].nunique()
    picks_per_draw = len(group) / n_draws
    p = picks_per_draw / n_pool
    exp = n_draws * p
    sd = np.sqrt(n_draws * p * (1 - p))
    band_rows.append({
        'Game': game,
        'Pool': n_pool,
        'Drawings': n_draws,
        'Expected': exp,
        'Lower Band': exp - 3 * sd,
        'Upper Band': exp + 3 * sd
    })

    game_freq = freq_current[freq_current['Game'] == game]
    hottest = game_freq.nlargest(1, 'Times Drawn').iloc[0]
    coldest = game_freq.nsmallest(1, 'Times Drawn').iloc[0]
    print(
        f'🎲 {game}: {n_draws:,} drawings, pool of {n_pool}, expected {exp:,.1f} per number')
    print(f'   🔥 hottest {int(hottest["Number"])} at {int(hottest["Times Drawn"]):,} | 🧊 coldest {int(coldest["Number"])} at {int(coldest["Times Drawn"]):,} | chance band {exp - 3 * sd:,.1f} to {exp + 3 * sd:,.1f}')

bands_df = pd.DataFrame(band_rows)
print(
    f'📉 Comparable drawings retained: {len(current_long):,} of {len(all_long):,} number picks')

🎲 Mega Millions: 675 drawings, pool of 70, expected 48.2 per number
   🔥 hottest 10 at 64 | 🧊 coldest 49 at 34 | chance band 28.1 to 68.3
🎲 Pick 10: 13,570 drawings, pool of 80, expected 3,392.5 per number
   🔥 hottest 58 at 3,541 | 🧊 coldest 25 at 3,250 | chance band 3,241.2 to 3,543.8
🎲 Powerball: 1,026 drawings, pool of 69, expected 74.3 per number
   🔥 hottest 61 at 97 | 🧊 coldest 13 at 55 | chance band 49.4 to 99.3
📉 Comparable drawings retained: 279,905 of 290,910 number picks


### **The Cost of Playing**

1. Define published game parameters sourced from official state lottery sites
2. Verify each jackpot probability independently using the combination formula
3. Calculate how long sustained play would take to reach a coin-flip chance at the jackpot
4. Calculate the opportunity cost of that same money placed in savings
5. Repeat the cost calculation using current 2026 ticket prices for the call to action

In [6]:
# Published game parameters, sourced from official state lottery sites
game_facts = {
    'Mega Millions': {
        'White Pool': 70, 'White Picks': 5, 'Bonus Pool': 25,
        'Ticket Price': 2.00, 'Draws Per Week': 2,
        'Era': 'Oct 2017 - Apr 2025, matches analysis window'
    },
    'Powerball': {
        'White Pool': 69, 'White Picks': 5, 'Bonus Pool': 26,
        'Ticket Price': 2.00, 'Draws Per Week': 3,
        'Era': 'Oct 2015 - present, matches analysis window'
    },
    'Pick 10': {
        'White Pool': 80, 'White Picks': 10, 'Bonus Pool': None,
        'Ticket Price': 1.00, 'Draws Per Week': 7,
        'Era': '1987 - present, unchanged'
    }
}

# Current ticket prices as of July 2026, Mega Millions rose to $5 in April 2025
current_prices = {'Mega Millions': 5.00, 'Powerball': 2.00, 'Pick 10': 1.00}

RETURN_RATE = 0.07
HORIZON_YEARS = 30

# Pick 10 draws 20 of 80, player matches all 10 of their picks
PICK10_JACKPOT_ODDS = comb(20, 10) / comb(80, 10)

cost_rows = []
for game, facts in game_facts.items():
    if facts['Bonus Pool'] is not None:
        combos = comb(
            facts['White Pool'],
            facts['White Picks']) * facts['Bonus Pool']
        p_jackpot = 1 / combos
    else:
        p_jackpot = PICK10_JACKPOT_ODDS
        combos = 1 / p_jackpot

    draws_per_year = facts['Draws Per Week'] * 52
    # Drawings needed for a cumulative 50% chance of at least one jackpot
    draws_to_coinflip = np.log(0.5) / np.log(1 - p_jackpot)
    years_to_coinflip = draws_to_coinflip / draws_per_year
    annual_spend = draws_per_year * facts['Ticket Price']

    cost_rows.append({
        'Game': game,
        'Jackpot Combinations': combos,
        'Odds Denominator': 1 / p_jackpot,
        'Draws Per Year': draws_per_year,
        'Ticket Price': facts['Ticket Price'],
        'Annual Spend': annual_spend,
        'Years To Coin Flip': years_to_coinflip,
        'Era': facts['Era']
    })

    print(f'💸 {game}: 1 in {1 / p_jackpot:,.0f} per ticket')
    print(f'   🎟️ playing every drawing costs ${annual_spend:,.0f} per year')
    print(
        f'   ⏳ {years_to_coinflip:,.0f} years of nonstop play for a 50/50 shot at the jackpot')

cost_df = pd.DataFrame(cost_rows)

# Future value of the same money invested instead of spent on tickets
growth_factor = ((1 + RETURN_RATE) ** HORIZON_YEARS - 1) / RETURN_RATE
cost_df['Savings At 30 Years'] = cost_df['Annual Spend'] * growth_factor

cost_df['Current Ticket Price'] = cost_df['Game'].map(current_prices)
cost_df['Current Annual Spend'] = cost_df['Draws Per Year'] * \
    cost_df['Current Ticket Price']
cost_df['Current Savings At 30 Years'] = cost_df['Current Annual Spend'] * growth_factor

for _, row in cost_df.iterrows():
    print(f'🏦 {row["Game"]} money redirected: ${row["Annual Spend"]:,.0f} per year becomes ${row["Savings At 30 Years"]:,.0f} after {HORIZON_YEARS} years at {RETURN_RATE:.0%}')

total_current = cost_df['Current Annual Spend'].sum()
total_savings = cost_df['Current Savings At 30 Years'].sum()

print(
    f'🎯 Playing all three games every drawing in 2026: ${total_current:,.0f} per year')
print(
    f'🎯 That same money in savings for {HORIZON_YEARS} years at {RETURN_RATE:.0%}: ${total_savings:,.0f}')
print(f'📎 Odds verified independently via the combination formula, not copied from a source')

💸 Mega Millions: 1 in 302,575,350 per ticket
   🎟️ playing every drawing costs $208 per year
   ⏳ 2,016,627 years of nonstop play for a 50/50 shot at the jackpot
💸 Powerball: 1 in 292,201,338 per ticket
   🎟️ playing every drawing costs $312 per year
   ⏳ 1,298,324 years of nonstop play for a 50/50 shot at the jackpot
💸 Pick 10: 1 in 8,911,711 per ticket
   🎟️ playing every drawing costs $364 per year
   ⏳ 16,970 years of nonstop play for a 50/50 shot at the jackpot
🏦 Mega Millions money redirected: $208 per year becomes $19,648 after 30 years at 7%
🏦 Powerball money redirected: $312 per year becomes $29,472 after 30 years at 7%
🏦 Pick 10 money redirected: $364 per year becomes $34,384 after 30 years at 7%
🎯 Playing all three games every drawing in 2026: $1,196 per year
🎯 That same money in savings for 30 years at 7%: $112,975
📎 Odds verified independently via the combination formula, not copied from a source


### **Preparing Data for Power BI**

1. Attach expected values and chance bands to the frequency table so each number carries its own reference range
2. Flag whether each number falls inside the range explained by chance
3. Build per-year frequency data restricted to the current matrix era for the small multiples visual
4. Export lifetime frequency tables to CSV for import into Power BI

In [7]:
# Prep for PowerBI
EXPORT_DIR = Path('powerbi_exports')
EXPORT_DIR.mkdir(exist_ok=True)

# Attach each game's expected value and chance band to every number
freq_export = freq_current.merge(bands_df, on='Game', how='left')
freq_export['Deviation'] = freq_export['Times Drawn'] - freq_export['Expected']
freq_export['Percent From Expected'] = (
    freq_export['Deviation'] / freq_export['Expected']) * 100
freq_export['Within Chance Band'] = (
    (freq_export['Times Drawn'] >= freq_export['Lower Band']) &
    (freq_export['Times Drawn'] <= freq_export['Upper Band'])
)

# Per-year frequency, current matrix era only
freq_yearly_current = (
    current_long.groupby(['Game', 'Year', 'Number'])
    .size()
    .reset_index(name='Times Drawn')
)

# Write everything out
exports = {
    'number_frequency.csv': freq_export,
    'frequency_by_year.csv': freq_yearly_current,
    'cost_of_playing.csv': cost_df
}

for filename, df in exports.items():
    df.to_csv(EXPORT_DIR / filename, index=False)
    print(f'📤 {filename}: {len(df):,} rows, {df.shape[1]} columns')

outside_band = freq_export[~freq_export['Within Chance Band']]
print(
    f'🧾 Numbers falling outside the chance band across all three games: {len(outside_band)}')

📤 number_frequency.csv: 219 rows, 11 columns
📤 frequency_by_year.csv: 4,257 rows, 4 columns
📤 cost_of_playing.csv: 3 rows, 12 columns
🧾 Numbers falling outside the chance band across all three games: 0


### **Annual Top Number**

1. Identify the most frequently drawn number in each game for each year
2. Retain every number tied for the top spot rather than breaking ties arbitrarily
3. Count how many distinct numbers have held the annual top spot in each game
4. Export the results for the hot number visualization

In [8]:
# Annual Top Number
year_max = freq_yearly_current.groupby(['Game', 'Year'])[
    'Times Drawn'].transform('max')
top_by_year = freq_yearly_current[freq_yearly_current['Times Drawn'] == year_max].copy(
)
top_by_year['Tied Numbers'] = top_by_year.groupby(
    ['Game', 'Year'])['Number'].transform('size')

top_by_year.to_csv(EXPORT_DIR / 'annual_top_number.csv', index=False)

for game, group in top_by_year.groupby('Game'):
    n_years = group['Year'].nunique()
    n_distinct = group['Number'].nunique()
    pool = freq_yearly_current[freq_yearly_current['Game']
                               == game]['Number'].max()
    print(f'🔥 {game}: {n_distinct} different numbers have topped a year across {n_years} years, out of a pool of {pool}')

repeat_toppers = (
    top_by_year.groupby(['Game', 'Number'])['Year']
    .nunique()
    .reset_index(name='Years On Top')
)
for game, group in repeat_toppers.groupby('Game'):
    repeats = group[group['Years On Top'] > 1]
    print(
        f'🔁 {game}: {len(repeats)} numbers have topped more than one year, most by any single number is {group["Years On Top"].max()}')
print(f'🔢 Rows per game: {top_by_year.groupby("Game").size().to_dict()}')
print(f'📤 annual_top_number.csv: {len(top_by_year):,} rows')
print(f'📁 Exports written to {EXPORT_DIR.resolve()}')

🔥 Mega Millions: 9 different numbers have topped a year across 8 years, out of a pool of 70
🔥 Pick 10: 35 different numbers have topped a year across 38 years, out of a pool of 80
🔥 Powerball: 17 different numbers have topped a year across 10 years, out of a pool of 69
🔁 Mega Millions: 0 numbers have topped more than one year, most by any single number is 1
🔁 Pick 10: 13 numbers have topped more than one year, most by any single number is 3
🔁 Powerball: 2 numbers have topped more than one year, most by any single number is 2
🔢 Rows per game: {'Mega Millions': 9, 'Pick 10': 49, 'Powerball': 19}
📤 annual_top_number.csv: 77 rows
📁 Exports written to C:\Users\slimt\DSC640\powerbi_exports


<br><br><br>

### **The Visual Story**

<img src="viz1_scatter_deviation.png" width="100%">

**Across all three games, every number's draw count sits within the range chance predicts. Nothing is hot, nothing is cold.**

<img src="viz2_hist_deviation.png" width="100%">

**Standardizing all 219 numbers onto one scale produces an ordinary bell curve. Not one crosses three standard deviations.**

<img src="viz3_topnumber_pick10.png" width="100%">

**Across 38 years of Pick 10, 35 different numbers have been the most-drawn of their year. Thirteen have been repeated, and chance alone predicts about ten.**

<img src="viz4_smallmultiples_yearly.png" width="100%">

**Ten years of Powerball, each panel flat. The lifetime totals are not hiding hot years that cancel out.**

<img src="viz5_bubble_odds_spend.png" width="100%">

**Pick 10 offers odds 34 times better than Powerball and still costs more per year. Higher cost buys nothing.**

<img src="viz6_bar_spend_savings.png" width="100%">

<span class="tex2jax_ignore mathjax_ignore">**Playing all three games every drawing costs $1,196 a year. That same money, saved at 7 percent for 30 years, becomes $112,975.**</span>

### **Summary**

This story is built for a general public audience of lottery players, people who buy tickets without a background in probability and who may believe that some numbers run hot. Because they are unfamiliar with the data, I kept technical vocabulary out of every chart title and moved terms like standard deviation into axis labels and captions, where a curious reader can find them without being stopped by them.

My purpose is persuasive rather than informational. The first four visuals establish that the drawings are clean, working from the full number pool down to individual years, so that no reasonable objection about hidden patterns is left standing. The final two turn that finding into a decision by showing what sustained play actually costs. The call to action is direct: redirect the money spent on tickets into savings, starting with a single game or a single drawing each week.

The medium is a visual deck built in Power BI, chosen because the argument is cumulative and needs a controlled sequence. An infographic would present all six findings at once and let the reader start anywhere, which would weaken the setup.

Design choices lean on Gestalt principles. Similarity drives a consistent color assignment, so the same three colors always mean the same three games, which lets most charts drop their legends. The histogram uses neutral gray instead, since its bars encode no category and color there would falsely suggest one. Proximity handles annotation, and I removed caption borders so whitespace does the grouping rather than added lines. All reference lines share one treatment, dark gray and dashed, so they read as a family distinct from the data. The palette is colorblind safe throughout. Titles use consistent title case and carry the finding rather than describing the chart, following Knaflic (2015).

Several transformations were required. Raw counts were standardized into standard deviations so games with very different draw volumes could be compared. Each game was restricted to its current matrix window, which is why Pick 10 spans 38 years while Mega Millions spans 8 and Powerball spans 10. Two visuals were filtered to a single game, and both state that filter on the chart. Partial years, a manually bounded histogram axis, and a bubble chart axis beginning at 200 rather than zero are each disclosed.

The data is published by the New York State Gaming Commission through data.ny.gov, contains no personal information, and carries no licensing restriction. Jackpot odds were verified independently using the combination formula rather than copied from a promotional source. The financial projection assumes a 7 percent annual return over 30 years, an assumption I chose and stated on the chart.

The clearest risk is that a reader concludes the games are fair and therefore worth playing more. Cairo (2016) argues that disclosure separates honest simplification from distortion, so I paired every fairness finding with its cost, and the deck closes on the cost rather than the fairness.

---

**References**

Cairo, A. (2016). *The truthful art: Data, charts, and maps for communication*. New Riders.

Knaflic, C. N. (2015). *Storytelling with data: A data visualization guide for business professionals*. Wiley.